# R3 - Exploratory P300 candidate measurements

Select the repository `.venv` kernel, Restart Kernel, then Run All.
This notebook measures subject-averaged ERPs separately for target and
distractor at Pz/CPz/Cz. Pz is the primary electrode. It does not repeat
cleaning, identify every maximum as P300, or automatically exclude people.

**Exploratory method for team review:** maximum signed voltage in a fixed
300-600 ms interval, with boundary and other quality flags. These settings
are a proposal in `erp.p300`, not a finalized report method.
Read the P300 section in `reports/r3_review_and_handoff.md` before reporting the values.


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from eegproj.config import find_root, load_config
from eegproj.evaluation.erp import collect_subject_erps
from eegproj.evaluation.p300 import (
    measure_candidate_peaks,
    plot_candidate_distribution,
    plot_candidate_review,
    summarise_candidate_peaks,
)

ROOT = find_root()
CONFIG = load_config(ROOT)
CHANNELS = CONFIG["erp"]["channels"]
PEAK_CONFIG = CONFIG["erp"]["p300"]
OUTPUT_DIR = ROOT / "results" / "erp"
FIGURE_DIR = ROOT / CONFIG["paths"]["figures"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load each included subject

Average trials within each subject and condition. Load subjects in turn;
do not pool all subjects' trials or measure only the grand-average peak.
Counts, epoch parameters and channel names are checked by the shared loader.


In [ ]:
subjects, times, subject_erps, epoch_counts = collect_subject_erps(ROOT, CONFIG, CHANNELS)
print("Included subjects:", len(subjects))
display(epoch_counts)

## 2. Candidate peak amplitude and latency

Take the largest signed sample in the fixed configured interval. Units:
amplitude in microvolts, latency in milliseconds from stimulus onset.
No additional filtering, interpolation or subject-specific window selection.
At 250 Hz the recorded-sample resolution is 4 ms.

Flag nonpositive maxima, exact or near-boundary maxima, non-local maxima,
and tied maxima. Retain every candidate in the data, including flagged
rows. A row without a peak-rule flag still needs visual signal QC from R2.
The fixed-window mean amplitude is exported as an additional descriptor.


In [ ]:
candidates = measure_candidate_peaks(subjects, times, subject_erps, epoch_counts, CHANNELS, CONFIG)
summary = summarise_candidate_peaks(candidates)
primary = candidates.loc[candidates["channel"].eq(PEAK_CONFIG["primary_channel"])].copy()
expected_rows = len(subjects) * len(PEAK_CONFIG["conditions"]) * len(CHANNELS)
assert len(candidates) == expected_rows
assert not candidates.duplicated(["subject", "condition", "channel"]).any()
assert np.isfinite(candidates[["candidate_peak_uv", "candidate_latency_ms", "mean_window_uv"]]).all().all()
display(summary)
print("Primary-electrode peak flags, independently of R2 visual QC:")
display(
    primary.groupby("condition")["peak_needs_review"]
    .agg(["size", "sum"])
    .rename(columns={"size": "n_subjects", "sum": "n_peak_review"})
)
display(
    primary.loc[
        primary["peak_needs_review"],
        ["subject", "condition", "candidate_peak_uv", "candidate_latency_ms", "peak_flags"],
    ]
)

## 3. Save descriptive tables and provenance

The full table contains subject x condition x channel rows (currently
35 x 2 x 3 = 210). These 210 rows are not 210 independent participants.
The primary paired table has one row per subject, with target and distractor
columns. Summary statistics include flagged candidates and show flag counts.


In [ ]:
candidates.to_csv(OUTPUT_DIR / "p300_candidate_peaks.csv", index=False)
summary.to_csv(OUTPUT_DIR / "p300_candidate_summary.csv", index=False)
candidates.loc[candidates["peak_needs_review"]].to_csv(OUTPUT_DIR / "p300_candidate_review.csv", index=False)
paired = primary.pivot(
    index="subject", columns="condition", values=["candidate_peak_uv", "candidate_latency_ms", "peak_needs_review"]
)
paired.columns = [f"{condition}_{metric}" for metric, condition in paired.columns]
paired.reset_index().to_csv(OUTPUT_DIR / "p300_primary_paired.csv", index=False)
manifest = {
    "measurement_status": PEAK_CONFIG["status"],
    "pipeline_version": 2,
    "subjects": subjects,
    "n_subjects": len(subjects),
    "channels": CHANNELS,
    "settings": PEAK_CONFIG,
    "algorithm": "maximum signed recorded ERP sample; earliest sample if tied; no extra smoothing or interpolation",
    "mean_amplitude": "mean voltage in the same fixed interval",
    "recorded_window": candidates[["first_sample_s", "last_sample_s", "n_window_samples", "sample_resolution_ms"]]
    .iloc[0]
    .to_dict(),
    "primary_peak_review": primary.groupby("condition")
    .agg(n_subjects=("subject", "nunique"), n_peak_review=("peak_needs_review", "sum"))
    .to_dict("index"),
    "visual_qc_status": epoch_counts["visual_review"].value_counts().to_dict(),
    "subject_exclusions": "no additional exclusions beyond the released v2 inclusion log",
    "interpretation": "candidate maximum is not automatic confirmation of a P300 component",
    "method_notes": "reports/r3_review_and_handoff.md",
}
(OUTPUT_DIR / "p300_candidate_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved candidate tables and manifest to:", OUTPUT_DIR)
display(paired.head())

## 4. Primary-electrode distributions

Each connected pair is one subject, not one trial or one electrode.
Orange crosses require peak review; blue circles have no peak-rule flag.
Both groups remain in the plot. No hypothesis test or group exclusion is
performed here, and signal QC approval remains pending separately.


In [ ]:
fig = plot_candidate_distribution(candidates, CONFIG)
for suffix in ("png", "pdf"):
    path = FIGURE_DIR / f"F2_p300_candidate_distribution.{suffix}"
    fig.savefig(path, dpi=CONFIG["erp"]["figure_dpi"], bbox_inches="tight")
plt.show()

## 5. Review individual Pz waveforms

The shaded interval is the candidate-search interval. Circles/crosses
mark target and distractor candidates. Subject titles with \* have at least
one peak-rule flag at Pz. Amplitude scales vary across panels for inspection;
use the paired plot and tables for numerical comparisons.

A near-boundary peak may reflect a later or earlier response, noise, or an
unsuitable window. Inspect the full waveform and discuss a uniform method
with the team rather than moving the window for just that subject.


In [ ]:
fig = plot_candidate_review(subjects, times, subject_erps, candidates, CHANNELS, CONFIG)
for suffix in ("png", "pdf"):
    path = FIGURE_DIR / f"F2_p300_candidate_review.{suffix}"
    fig.savefig(path, dpi=CONFIG["erp"]["figure_dpi"], bbox_inches="tight")
plt.show()

## Next step

Review the flagged candidates and agree on the fixed measurement rule with
the team before finalizing report values. Coordinate signal QC with R2.
Afterwards, prepare the behavioral summary (hit rate, false alarms, RT)
from the preprocessing log and draft Results - ERP.

Generated figures, CSV files and the manifest are gitignored. Keep source
notebooks stripped when committing and share outputs via the team folder.
